In [23]:
import os
import json
import time
import copy
from pathlib import Path
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import RobustScaler, KBinsDiscretizer
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

from kneed import KneeLocator

import optuna
from optuna.importance import get_param_importances
import optuna.visualization as vis

from NN_model import ImprovedNN

#Import the model and helper functions
from NN_model import ImprovedNN
from NN_updated_transferlearning_copy import parse_embeddings, find_optimal_clusters, plot_cluster_tSNE, plot_cluster_distribution, bin_mp_values, create_stratified_folds, plot_fold_histograms, preload_and_scale_all_folds, EarlyStopper, plot_training_progress, evaluate_fold, objective, set_optuna_study, train_and_save_best_general_models, train_full_test_and_eval  
                                      

In [24]:
# 1) Define the base folder *relative* to your notebook/script
current_Path = Path.cwd()
BASE = current_Path.parent.parent


# 2) Import the train and test file
test_file_dir = BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "test_predictions" / "consensus_without_data_augmentation.csv"
train_file_dir = BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "train_without_data_augmentation" / "final_train_df.csv"

# 3) Load dataframe 
test_df = pd.read_csv(test_file_dir)
train_df = pd.read_csv(train_file_dir)

In [25]:
test_df["MW"].describe()

count    1961.000000
mean      239.068269
std       107.479056
min        18.015000
25%       165.192000
50%       215.043000
75%       299.282000
max       922.572000
Name: MW, dtype: float64

In [26]:
t1, t2 = 250, 500
print(f"Ranges (based on TRAIN): Low ≤ {t1}, Intermediate ({t1}, {t2}], High > {t2}")

def mw_default_thresholds(mw_series, t1=250, t2=500):
    return np.select(
        [mw_series <= t1, mw_series > t2],
        ["Low", "High"],
        default="Intermediate"
    )

train_df["MW_category_default"] = mw_default_thresholds(train_df["MW"], t1, t2)
test_df["MW_category_default"]  = mw_default_thresholds(test_df["MW"],  t1, t2)

Ranges (based on TRAIN): Low ≤ 250, Intermediate (250, 500], High > 500


In [27]:
print(train_df["MW_category_default"].value_counts(dropna=False))
print(test_df["MW_category_default"].value_counts(dropna=False))




MW_category_default
Low             11006
Intermediate     6217
High              410
Name: count, dtype: int64
MW_category_default
Low             1235
Intermediate     672
High              54
Name: count, dtype: int64


In [28]:
def freeze_hidden_layers(model, freeze_config):
    print(f"Applying freeze configuration: {freeze_config}")
    
    for layer_index, should_freeze in enumerate(freeze_config):
        if should_freeze == 1:  # Freeze this layer
            # Each hidden layer consists of 4 components: Linear, BatchNorm, ReLU, Dropout
            block_start = layer_index * 4
            block_end = block_start + 4
            
            print(f"  Freezing layer {layer_index + 1} (network indices {block_start}-{block_end-1})")
            
            # Freeze all parameters in this layer block
            for i in range(block_start, min(block_end, len(model.network))):
                for param in model.network[i].parameters():
                    param.requires_grad = False
    
    return model

In [32]:
def transfer_learning_category_pipeline(data, category_name, pretrained_model_path=None, freeze_config=[0, 0, 0], save_dir="transfer_learning_models", model_suffix="",
                                      n_trials=1, n_folds=10, max_epochs=100, patience=12, device="mps", random_state=42, mw_category_col="MW_category_default"):
    
    print(f"\n{'='*80}")
    print(f"TRANSFER LEARNING PIPELINE: {category_name.upper()} MW CATEGORY")
    print(f"{'='*80}")
    
    device = torch.device(device)
    
    # Step 1: Filter data by MW category
    print(f"\nStep 1: Filtering data for {category_name} MW category")
    filtered_data = data[data[mw_category_col] == category_name].copy().reset_index(drop=True)
    print(f"Original data distribution: {data[mw_category_col].value_counts()}")
    print(f"Selected {len(filtered_data)} samples for {category_name} category")
    
    if len(filtered_data) < 50:
        print(f"Warning: Only {len(filtered_data)} samples available for {category_name}. Consider combining categories.")
        return None
    
    # Step 2: Parse embeddings and create structure clusters
    print(f"\nStep 2: Parsing embeddings and creating structure clusters")
    X, y = parse_embeddings(filtered_data, emb_col="embeddings", target_col="MP")
    print(f"Data shape: X={X.shape}, y={y.shape}")
    
    # Find optimal clusters and apply clustering
    k_opt, wcss, k_list = find_optimal_clusters(X, max_k=min(15, len(filtered_data)//5), plot=True, random_state=random_state)
    final_kmeans = KMeans(n_clusters=k_opt, init="k-means++", n_init=10, random_state=random_state)
    filtered_data["Structure_Cluster"] = final_kmeans.fit_predict(X)
    
    # Visualize clusters
    plot_cluster_tSNE(X, filtered_data["Structure_Cluster"], title=f"t-SNE Clustering - {category_name.capitalize()} MW", perplexity=min(30, len(filtered_data)//4))
    plot_cluster_distribution(filtered_data, cluster_col="Structure_Cluster", title=f"Cluster Distribution - {category_name.capitalize()} MW")
    
    # Step 3: Create MP bins and stratified folds
    print(f"\nStep 3: Creating MP bins and stratified folds")
    filtered_data = bin_mp_values(filtered_data, mp_col="MP", n_bins=4, strategy='quantile')
    
    # Create temporary save path for folds
    temp_fold_path = Path(save_dir) / f"temp_{category_name}_folds.csv"
    filtered_data_with_folds, folds = create_stratified_folds(filtered_data, save_path=temp_fold_path, n_splits=n_folds, random_state=random_state)
    
    # Visualize fold distributions
    print(f"\nVisualizing fold distributions for {category_name}...")
    mp_bins = np.histogram_bin_edges(filtered_data_with_folds["MP"].dropna().to_numpy(), bins=30)
    cluster_levels = np.sort(filtered_data_with_folds["Structure_Cluster"].dropna().unique())
    
    # Show distributions for folds 
    for f in range(n_folds):
        plot_fold_histograms(filtered_data_with_folds, f, mp_bins=mp_bins, cluster_levels=cluster_levels)
    
    # Step 4: Prepare fold data for cross-validation
    print(f"\nStep 4: Preparing fold data for {n_folds}-fold cross-validation")
    fold_data = {f["fold"]: {"train": filtered_data_with_folds.iloc[f["train_idx"]].copy(), "val": filtered_data_with_folds.iloc[f["val_idx"]].copy()} for f in folds}
    
    train_X, train_y, val_X, val_y, fold_scalers = preload_and_scale_all_folds(fold_data)
    
    # Step 5: Hyperparameter optimization with cross-validation
    print(f"\nStep 5: Hyperparameter optimization with {n_folds}-fold cross-validation")
    print(f"Running {n_trials} trials for {category_name} category...")
    
    def objective(trial):
        fold_rmses = []
        
        # Get hyperparameters
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        
        for fold in range(n_folds):
            # Create model - either from pretrained or new
            if pretrained_model_path and Path(pretrained_model_path).exists():
                # Load pretrained model
                checkpoint = torch.load(pretrained_model_path, map_location=device)
                input_size = checkpoint.get('input_size', X.shape[1])
                
                # Create model with same architecture but new dropout rate
                model = ImprovedNN(input_size=input_size, dropout_rate=dropout_rate).to(device)
                
                # Load pretrained weights
                try:
                    model.load_state_dict(checkpoint['model_state_dict'])
                    print(f"Loaded pretrained model from {pretrained_model_path}")
                except:
                    print(f"Warning: Could not load pretrained weights, using random initialization")
                
                # Apply freeze configuration
                model = freeze_hidden_layers(model, freeze_config)
            else:
                # Create new model from scratch
                model = ImprovedNN(input_size=X.shape[1], dropout_rate=dropout_rate).to(device)
            
            # Train single fold using your existing evaluate_fold function
            rmse, _, _, _, _, _, _ = evaluate_fold(trial=trial, fold_idx=fold, X_train_scaled=train_X[fold], y_train=train_y[fold], 
                X_val_scaled=val_X[fold], y_val=val_y[fold], learning_rate=learning_rate, batch_size=batch_size, 
                dropout_rate=dropout_rate, weight_decay=weight_decay, max_epochs=max_epochs, patience=patience)
            fold_rmses.append(rmse)
        
        return np.mean(fold_rmses)
    
    # Run optimization
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, timeout=None)
    
    best_params = study.best_params
    print(f"\nBest hyperparameters for {category_name}: {best_params}")
    print(f"Best CV RMSE: {study.best_value:.4f}")
    
    # Step 6: Train and save 10 fold models with best hyperparameters
    print(f"\nStep 6: Training and saving {n_folds} fold models with best hyperparameters")
    
    # Create save directory
    category_save_dir = Path(save_dir) / category_name
    category_save_dir.mkdir(parents=True, exist_ok=True)
    
    fold_results = []
    
    for fold in range(n_folds):
        print(f"\n--- Training final model for {category_name} Fold {fold} ---")
        
        # Create model with best hyperparameters
        if pretrained_model_path and Path(pretrained_model_path).exists():
            checkpoint = torch.load(pretrained_model_path, map_location=device)
            input_size = checkpoint.get('input_size', X.shape[1])
            model = ImprovedNN(input_size=input_size, dropout_rate=best_params['dropout_rate']).to(device)
            
            try:
                model.load_state_dict(checkpoint['model_state_dict'])
                model = freeze_hidden_layers(model, freeze_config)
            except:
                print(f"Warning: Could not load pretrained weights for fold {fold}")
        else:
            model = ImprovedNN(input_size=X.shape[1], dropout_rate=best_params['dropout_rate']).to(device)
        
        # Train model
        rmse, r2, q2, model, train_losses, val_losses, stop_epoch = evaluate_fold(
            trial=None, fold_idx=fold,
            X_train_scaled=train_X[fold], y_train=train_y[fold], 
            X_val_scaled=val_X[fold], y_val=val_y[fold],
            learning_rate=best_params["learning_rate"], batch_size=best_params["batch_size"],
            dropout_rate=best_params["dropout_rate"], weight_decay=best_params["weight_decay"],
            max_epochs=max_epochs, patience=patience)
        
        # Plot training progress
        plot_training_progress(train_losses, val_losses, early_stop_epoch=stop_epoch, title=f"{category_name} - Fold {fold} Loss Curve")
        
        # Save fold model
        model_name = f"fold{fold}_{model_suffix}" if model_suffix else f"fold{fold}"
        model_path = category_save_dir / f"{model_name}.pt"
        
        torch.save({"model_state_dict": model.state_dict(), "input_size": X.shape[1], 
            "hyperparams": dict(best_params), "metrics": {"rmse": float(rmse), "r2": float(r2), "q2": float(q2)}, 
            "fold": fold,"category": category_name, "freeze_config": freeze_config,
            "pretrained_from": pretrained_model_path, "scaler": fold_scalers[fold]}, model_path)
        
        print(f"Saved {category_name} fold {fold} model to: {model_path}")
        
        fold_results.append({ "Fold": fold,  "Category": category_name, "Train_Size": int(len(train_y[fold])),
            "Val_Size": int(len(val_y[fold])), "Val_RMSE": float(rmse), 
            "Val_R2": float(r2), "Val_Q2": float(q2), "Model_Path": str(model_path)})
    
    # Save fold results summary
    fold_results_df = pd.DataFrame(fold_results)
    summary_path = category_save_dir / f"{category_name}_fold_summary.csv"
    fold_results_df.to_csv(summary_path, index=False)
    print(f"\nSaved fold summary to: {summary_path}")
    
    # Step 7: Train final test model using full category data and best hyperparameters
    print(f"\nStep 7: Training final test model for {category_name} using best hyperparameters")
    
    # Re-split the filtered data for final model training
    tr_df, val_df = train_test_split(filtered_data_with_folds, test_size=0.20, random_state=random_state, 
        stratify=filtered_data_with_folds["Stratify_Label"])
    print(f"Final model - Train rows: {len(tr_df)}, Val rows: {len(val_df)}")
    
    # Parse and scale data for final model
    X_tr_raw, y_tr = parse_embeddings(tr_df, emb_col="embeddings", target_col="MP")
    X_val_raw, y_val = parse_embeddings(val_df, emb_col="embeddings", target_col="MP")
    
    final_scaler = RobustScaler().fit(X_tr_raw)
    X_tr_scaled = final_scaler.transform(X_tr_raw)
    X_val_scaled = final_scaler.transform(X_val_raw)
    
    # Create final model
    if pretrained_model_path and Path(pretrained_model_path).exists():
        checkpoint = torch.load(pretrained_model_path, map_location=device)
        input_size = checkpoint.get('input_size', X.shape[1])
        final_model = ImprovedNN(input_size=input_size, dropout_rate=best_params['dropout_rate']).to(device)
        
        try:
            final_model.load_state_dict(checkpoint['model_state_dict'])
            final_model = freeze_hidden_layers(final_model, freeze_config)
            print(f"Final model initialized from pretrained weights with freeze config: {freeze_config}")
        except:
            print("Warning: Could not load pretrained weights for final model")
    else:
        final_model = ImprovedNN(input_size=X.shape[1], dropout_rate=best_params['dropout_rate']).to(device)
        print("Final model created from scratch")
    
    # Training setup
    criterion = nn.HuberLoss(delta=0.5)
    optimizer = optim.AdamW(final_model.parameters(), lr=best_params['learning_rate'], 
                           weight_decay=best_params['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    # Data loaders
    train_dataset = TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32).to(device), torch.tensor(y_tr, dtype=torch.float32).to(device))
    val_dataset = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32).to(device), torch.tensor(y_val, dtype=torch.float32).to(device))
    
    train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=best_params['batch_size'], shuffle=False, num_workers=0)
    
    # Training loop with early stopping
    early_stopper = EarlyStopper(patience=patience, min_delta=0)
    best_val_loss = float('inf')
    best_model_state = None
    train_losses, val_losses = [], []
    
    for epoch in range(1, max_epochs + 1):
        # Training
        final_model.train()
        epoch_train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = final_model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), 0.5)
            optimizer.step()
            epoch_train_loss += loss.item()
        
        epoch_train_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        
        # Validation
        final_model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = final_model(xb)
                loss = criterion(preds, yb)
                epoch_val_loss += loss.item()
        
        epoch_val_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        scheduler.step(epoch_val_loss)
        
        # Track best model
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_state = copy.deepcopy(final_model.state_dict())
        
        # Early stopping
        if early_stopper.early_stop(epoch_val_loss, epoch=epoch):
            print(f"Early stopping at epoch {epoch}")
            break
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:4d}, Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")
    
    # Load best model and calculate final metrics
    if best_model_state:
        final_model.load_state_dict(best_model_state)
    
    # Calculate final metrics
    final_model.eval()
    with torch.no_grad():
        preds_val = torch.cat([final_model(xb).cpu() for xb, _ in val_loader]).numpy()
    
    final_metrics = {'rmse': root_mean_squared_error(y_val, preds_val),'r2': r2_score(y_val, preds_val), 'q2': 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_tr.mean())**2)}
    
    print(f"Final {category_name} model metrics - RMSE: {final_metrics['rmse']:.4f}, R²: {final_metrics['r2']:.4f}, Q²: {final_metrics['q2']:.4f}")
    
    # Plot final training progress
    plot_training_progress(train_losses, val_losses, early_stop_epoch=early_stopper.stop_epoch, title=f"{category_name} Final Model Training Progress")
    
    # Save final test model
    final_model_name = f"final_test_model_{model_suffix}" if model_suffix else "final_test_model"
    final_model_path = category_save_dir / f"{final_model_name}.pt"
    
    checkpoint = {'model_state_dict': final_model.state_dict(),'scaler': final_scaler,'metrics': final_metrics,
        'best_params': best_params,'input_size': X.shape[1],'dropout_rate': best_params['dropout_rate'],
        'category': category_name,'freeze_config': freeze_config,'pretrained_from': pretrained_model_path,
        'n_samples': len(filtered_data),'model_class': 'ImprovedNN'}
    
    torch.save(checkpoint, final_model_path)
    print(f"Saved final {category_name} model to: {final_model_path}")
    
    # Clean up temporary fold file
    if temp_fold_path.exists():
        temp_fold_path.unlink()
    
    # Return comprehensive results
    return {'category': category_name, 'final_model_path': str(final_model_path),'fold_models_dir': str(category_save_dir),
        'metrics': final_metrics, 'best_params': best_params,'n_samples': len(filtered_data),
        'freeze_config': freeze_config,'pretrained_from': pretrained_model_path,'fold_results': fold_results_df,
        'cv_rmse': study.best_value}




In [33]:
def transfer_learning_category_pipeline(data, category_name, pretrained_model_path=None, freeze_config=[0, 0, 0], save_dir="transfer_learning_models", model_suffix="",
                                      n_trials=50, n_folds=10, max_epochs=100, patience=12, device="mps", random_state=42, mw_category_col="MW_category_quant"):

    print(f"\n{'='*80}")
    print(f"TRANSFER LEARNING PIPELINE: {category_name.upper()} MW CATEGORY")
    print(f"{'='*80}")
    
    device = torch.device(device)
    
    # Step 1: Filter data by MW category
    print(f"\nStep 1: Filtering data for {category_name} MW category")
    filtered_data = data[data[mw_category_col] == category_name].copy().reset_index(drop=True)
    print(f"Original data distribution: {data[mw_category_col].value_counts()}")
    print(f"Selected {len(filtered_data)} samples for {category_name} category")
    
    if len(filtered_data) < 50:
        print(f"Warning: Only {len(filtered_data)} samples available for {category_name}. Results might be skewed.")
        return None
    
    # Step 2: Parse embeddings and create structure clusters
    print(f"\nStep 2: Parsing embeddings and creating structure clusters")
    X, y = parse_embeddings(filtered_data, emb_col="embeddings", target_col="MP")
    
    # Find optimal clusters and apply clustering
    k_opt, wcss, k_list = find_optimal_clusters(X, max_k=min(15), plot=True, random_state=random_state)
    final_kmeans = KMeans(n_clusters=k_opt, init="k-means++", n_init=10, random_state=random_state)
    filtered_data["Structure_Cluster"] = final_kmeans.fit_predict(X)
    
    # Visualize clusters
    plot_cluster_tSNE(X, filtered_data["Structure_Cluster"], title=f"t-SNE Clustering - {category_name.capitalize()} MW", perplexity=min(30, len(filtered_data)//4))
    plot_cluster_distribution(filtered_data, cluster_col="Structure_Cluster", title=f"Cluster Distribution - {category_name.capitalize()} MW")
    
    # Step 3: Create MP bins and stratified folds
    print(f"\nStep 3: Creating MP bins and stratified folds")
    filtered_data = bin_mp_values(filtered_data, mp_col="MP", n_bins=4, strategy='quantile')
    
    # Create temporary save path for folds
    temp_fold_path = Path(save_dir) / f"temp_{category_name}_folds.csv"
    filtered_data_with_folds, folds = create_stratified_folds(filtered_data, save_path=temp_fold_path, n_splits=n_folds, random_state=random_state)
    
    # Visualize fold distributions
    print(f"\nVisualizing fold distributions for {category_name}...")
    mp_bins = np.histogram_bin_edges(filtered_data_with_folds["MP"].dropna().to_numpy(), bins=30)
    cluster_levels = np.sort(filtered_data_with_folds["Structure_Cluster"].dropna().unique())
    
    # Show distributions for folds 
    for f in range((n_folds)):
        plot_fold_histograms(filtered_data_with_folds, f, mp_bins=mp_bins, cluster_levels=cluster_levels)
    
    # Step 4: Prepare fold data for cross-validation
    print(f"\nStep 4: Preparing fold data for {n_folds}-fold cross-validation")
    fold_data = {f["fold"]: { "train": filtered_data_with_folds.iloc[f["train_idx"]].copy(), "val": filtered_data_with_folds.iloc[f["val_idx"]].copy()} for f in folds}
    
    train_X, train_y, val_X, val_y, fold_scalers = preload_and_scale_all_folds(fold_data)
    
    # Step 5: Hyperparameter optimization with cross-validation
    print(f"\nStep 5: Hyperparameter optimization with {n_folds}-fold cross-validation")
    print(f"Running {n_trials} trials for {category_name} category...")
    
    def objective(trial):
        fold_rmses = []
        
        # Get hyperparameters
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        
        for fold in range(n_folds):
            # Create model - either from pretrained or new
            if pretrained_model_path and Path(pretrained_model_path).exists():
                # Load pretrained model
                checkpoint = torch.load(pretrained_model_path, map_location=device)
                input_size = checkpoint.get('input_size', X.shape[1])
                
                # Create model with same architecture but new dropout rate
                model = ImprovedNN(input_size=input_size, dropout_rate=dropout_rate).to(device)
                
                # Load pretrained weights
                try:
                    model.load_state_dict(checkpoint['model_state_dict'])
                    print(f"Loaded pretrained model from {pretrained_model_path}")
                except:
                    print(f"Warning: Could not load pretrained weights, using random initialization")
                
                # Apply freeze configuration
                model = freeze_hidden_layers(model, freeze_config)
            else:
                # Create new model from scratch
                model = ImprovedNN(input_size=X.shape[1], dropout_rate=dropout_rate).to(device)
            
            # Train single fold using your existing evaluate_fold function
            rmse, _, _, _, _, _, _ = evaluate_fold(trial=trial, fold_idx=fold, X_train_scaled=train_X[fold], y_train=train_y[fold], X_val_scaled=val_X[fold], y_val=val_y[fold],
                learning_rate=learning_rate, batch_size=batch_size, dropout_rate=dropout_rate, weight_decay=weight_decay, max_epochs=max_epochs, patience=patience)
            fold_rmses.append(rmse)
        
        return np.mean(fold_rmses)
    
    # Run optimization
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, timeout=None)
    
    best_params = study.best_params
    print(f"\nBest hyperparameters for {category_name}: {best_params}")
    print(f"Best CV RMSE: {study.best_value:.4f}")
    
    # Step 6: Train and save 10 fold models with best hyperparameters
    print(f"\nStep 6: Training and saving {n_folds} fold models with best hyperparameters")
    
    # Create save directory
    category_save_dir = Path(save_dir) / category_name
    category_save_dir.mkdir(parents=True, exist_ok=True)
    
    fold_results = []
    
    for fold in range(n_folds):
        print(f"\n--- Training final model for {category_name} Fold {fold} ---")
        
        # Create model with best hyperparameters
        if pretrained_model_path and Path(pretrained_model_path).exists():
            checkpoint = torch.load(pretrained_model_path, map_location=device)
            input_size = checkpoint.get('input_size', X.shape[1])
            model = ImprovedNN(input_size=input_size, dropout_rate=best_params['dropout_rate']).to(device)
            
            try:
                model.load_state_dict(checkpoint['model_state_dict'])
                model = freeze_hidden_layers(model, freeze_config)
            except:
                print(f"Warning: Could not load pretrained weights for fold {fold}")
        else:
            model = ImprovedNN(input_size=X.shape[1], dropout_rate=best_params['dropout_rate']).to(device)
        
        # Train model
        rmse, r2, q2, model, train_losses, val_losses, stop_epoch = evaluate_fold(trial=None, fold_idx=fold, X_train_scaled=train_X[fold], y_train=train_y[fold], 
            X_val_scaled=val_X[fold], y_val=val_y[fold], learning_rate=best_params["learning_rate"], batch_size=best_params["batch_size"],
            dropout_rate=best_params["dropout_rate"], weight_decay=best_params["weight_decay"],
            max_epochs=max_epochs, patience=patience)
        
        # Plot training progress
        plot_training_progress(train_losses, val_losses, early_stop_epoch=stop_epoch, title=f"{category_name} - Fold {fold} Loss Curve")
        
        # Save fold model
        model_name = f"fold{fold}_{model_suffix}" if model_suffix else f"fold{fold}"
        model_path = category_save_dir / f"{model_name}.pt"
        
        torch.save({"model_state_dict": model.state_dict(), "input_size": X.shape[1], "hyperparams": dict(best_params),"metrics": {"rmse": float(rmse), "r2": float(r2), "q2": float(q2)}, 
            "fold": fold, "category": category_name, "freeze_config": freeze_config, "pretrained_from": pretrained_model_path, "scaler": fold_scalers[fold]
        }, model_path)
        
        print(f"Saved {category_name} fold {fold} model to: {model_path}")
        
        fold_results.append({"Fold": fold, "Category": category_name, "Train_Size": int(len(train_y[fold])), "Val_Size": int(len(val_y[fold])), 
            "Val_RMSE": float(rmse), "Val_R2": float(r2), "Val_Q2": float(q2), "Model_Path": str(model_path)})
    
    # Save fold results summary
    fold_results_df = pd.DataFrame(fold_results)
    summary_path = category_save_dir / f"{category_name}_fold_summary.csv"
    fold_results_df.to_csv(summary_path, index=False)
    print(f"\nSaved fold summary to: {summary_path}")
    
    # Step 7: Train final test model using full category data and best hyperparameters
    print(f"\nStep 7: Training final test model for {category_name} using best hyperparameters")
    
    # Re-split the filtered data for final model training
    tr_df, val_df = train_test_split(filtered_data_with_folds, test_size=0.20, random_state=random_state, stratify=filtered_data_with_folds["Stratify_Label"])
    print(f"Final model - Train rows: {len(tr_df)}, Val rows: {len(val_df)}")
    
    # Parse and scale data for final model
    X_tr_raw, y_tr = parse_embeddings(tr_df, emb_col="embeddings", target_col="MP")
    X_val_raw, y_val = parse_embeddings(val_df, emb_col="embeddings", target_col="MP")
    
    final_scaler = RobustScaler().fit(X_tr_raw)
    X_tr_scaled = final_scaler.transform(X_tr_raw)
    X_val_scaled = final_scaler.transform(X_val_raw)
    
    # Create final model
    if pretrained_model_path and Path(pretrained_model_path).exists():
        checkpoint = torch.load(pretrained_model_path, map_location=device)
        input_size = checkpoint.get('input_size', X.shape[1])
        final_model = ImprovedNN(input_size=input_size, dropout_rate=best_params['dropout_rate']).to(device)
        
        try:
            final_model.load_state_dict(checkpoint['model_state_dict'])
            final_model = freeze_hidden_layers(final_model, freeze_config)
            print(f"Final model initialized from pretrained weights with freeze config: {freeze_config}")
        except:
            print("Warning: Could not load pretrained weights for final model")
    else:
        final_model = ImprovedNN(input_size=X.shape[1], dropout_rate=best_params['dropout_rate']).to(device)
        print("Final model created from scratch")
    
    # Training setup
    criterion = nn.HuberLoss(delta=0.5)
    optimizer = optim.AdamW(final_model.parameters(), lr=best_params['learning_rate'], 
                           weight_decay=best_params['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    # Data loaders
    train_dataset = TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32).to(device), torch.tensor(y_tr, dtype=torch.float32).to(device))
    val_dataset = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32).to(device), torch.tensor(y_val, dtype=torch.float32).to(device))
    
    train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=best_params['batch_size'], shuffle=False, num_workers=0)
    
    # Training loop with early stopping
    early_stopper = EarlyStopper(patience=patience, min_delta=0)
    best_val_loss = float('inf')
    best_model_state = None
    train_losses, val_losses = [], []
    
    for epoch in range(1, max_epochs + 1):
        # Training
        final_model.train()
        epoch_train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = final_model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), 0.5)
            optimizer.step()
            epoch_train_loss += loss.item()
        
        epoch_train_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        
        # Validation
        final_model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = final_model(xb)
                loss = criterion(preds, yb)
                epoch_val_loss += loss.item()
        
        epoch_val_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        scheduler.step(epoch_val_loss)
        
        # Track best model
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_state = copy.deepcopy(final_model.state_dict())
        
        # Early stopping
        if early_stopper.early_stop(epoch_val_loss, epoch=epoch):
            print(f"Early stopping at epoch {epoch}")
            break
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:4d}, Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")
    
    # Load best model and calculate final metrics
    if best_model_state:
        final_model.load_state_dict(best_model_state)
    
    # Calculate final metrics
    final_model.eval()
    with torch.no_grad():
        preds_val = torch.cat([final_model(xb).cpu() for xb, _ in val_loader]).numpy()
    
    final_metrics = {'rmse': root_mean_squared_error(y_val, preds_val), 'r2': r2_score(y_val, preds_val), 'q2': 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_tr.mean())**2)
}
    
    print(f"Final {category_name} model metrics - RMSE: {final_metrics['rmse']:.4f}, R²: {final_metrics['r2']:.4f}, Q²: {final_metrics['q2']:.4f}")
    
    # Plot final training progress
    plot_training_progress(train_losses, val_losses, early_stop_epoch=early_stopper.stop_epoch, title=f"{category_name} Final Model Training Progress")
    
    # Save final test model
    final_model_name = f"final_test_model_{model_suffix}" if model_suffix else "final_test_model"
    final_model_path = category_save_dir / f"{final_model_name}.pt"
    
    checkpoint = {
        'model_state_dict': final_model.state_dict(),
        'scaler': final_scaler,
        'metrics': final_metrics,
        'best_params': best_params,
        'input_size': X.shape[1],
        'dropout_rate': best_params['dropout_rate'],
        'category': category_name,
        'freeze_config': freeze_config,
        'pretrained_from': pretrained_model_path,
        'n_samples': len(filtered_data),
        'model_class': 'ImprovedNN'
    }
    
    torch.save(checkpoint, final_model_path)
    print(f"Saved final {category_name} model to: {final_model_path}")
    
    # Clean up temporary fold file
    if temp_fold_path.exists():
        temp_fold_path.unlink()
    
    # Return comprehensive results
    return {
        'category': category_name,
        'final_model_path': str(final_model_path),
        'fold_models_dir': str(category_save_dir),
        'metrics': final_metrics,
        'best_params': best_params,
        'n_samples': len(filtered_data),
        'freeze_config': freeze_config,
        'pretrained_from': pretrained_model_path,
        'fold_results': fold_results_df,
        'cv_rmse': study.best_value
    }

In [34]:
def run_complete_transfer_learning_pipeline(data, save_dir = "transfer_learning_models", n_trials = 1, max_epochs = 100,
                                          patience = 12, device = "mps", mw_category_col: str = "ct",
                                          random_state: int = 42):

    
    results = {}
    
    print("="*100)
    print("COMPLETE TRANSFER LEARNING PIPELINE")
    print("="*100)
    
    # Check available categories
    print("Available MW categories:")
    category_counts = data[mw_category_col].value_counts()
    print(category_counts)
    
    # Step 1: Train Low MW model (base model)
    print("\n" + "="*100)
    print("PHASE 1: TRAINING LOW MW MODEL (BASE MODEL)")
    print("="*100)
    
    low_result = transfer_learning_category_pipeline(data=data, category_name="Low", pretrained_model_path=None,  # Train from scratch
        freeze_config=[0, 0, 0], save_dir=save_dir, model_suffix="0_0_0", n_trials=n_trials,max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col,random_state=random_state)
    
  
    # Step 2: Train Intermediate MW models
    print("\n" + "="*100)
    print("PHASE 2: TRAINING INTERMEDIATE MW MODELS")
    print("="*100)
    
    # 2a: Intermediate from low 000 with NO freezing
    int_000 = transfer_learning_category_pipeline(data=data,category_name="Intermediate", pretrained_model_path=low_result['final_model_path'],  # Train from Low
        freeze_config=[0, 0, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)
    
    # 2b: Intermediate with low 000 with FREEZING 1,0,0
    int_100 = transfer_learning_category_pipeline(data=data,category_name="Intermediate", pretrained_model_path=low_result['final_model_path'],  # Train from Low
        freeze_config=[1, 0, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)
    
        # 2c: Intermediate with low 000 with FREEZING 1,1,0
    int_110 = transfer_learning_category_pipeline(data=data,category_name="Intermediate", pretrained_model_path=low_result['final_model_path'],  # Train from Low
        freeze_config=[1, 1, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)
    
    
    # Step 3: Train High MW models
    print("\n" + "="*100)
    print("PHASE 3: TRAINING HIGH MW MODELS")
    print("="*100)

     # 3a: High from Med 100 with NO freezing
    high_000 = transfer_learning_category_pipeline(data=data,category_name="High", pretrained_model_path=int_100['final_model_path'],  # Train from Low
        freeze_config=[0, 0, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)   
    
    # 3b: High from Med 100 with 1_0_0
    high_000 = transfer_learning_category_pipeline(data=data,category_name="High", pretrained_model_path=int_100['final_model_path'],  # Train from Low
        freeze_config=[1, 0, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)   
    
    # 3c: High from Med 100 with 1_1_0
    high_000 = transfer_learning_category_pipeline(data=data,category_name="High", pretrained_model_path=int_100['final_model_path'],  # Train from Low
        freeze_config=[1, 1, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state) 

     # 3a1: High from Med 100 with NO freezing
    high_000 = transfer_learning_category_pipeline(data=data,category_name="High", pretrained_model_path=int_110['final_model_path'],  # Train from Low
        freeze_config=[0, 0, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)   
    
    # 3b1: High from Med 100 with 1_0_0
    high_000 = transfer_learning_category_pipeline(data=data,category_name="High", pretrained_model_path=int_110['final_model_path'],  # Train from Low
        freeze_config=[1, 0, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state)   
    
    # 3c1: High from Med 100 with 1_1_0
    high_000 = transfer_learning_category_pipeline(data=data,category_name="High", pretrained_model_path=int_110['final_model_path'],  # Train from Low
        freeze_config=[1, 1, 0], save_dir=save_dir, model_suffix="000", n_trials=n_trials, max_epochs=max_epochs,
        patience=patience, device=device, mw_category_col=mw_category_col, random_state=random_state) 
    
    # Print comprehensive summary
    print("\n" + "="*100)
    print("TRANSFER LEARNING PIPELINE COMPLETE - COMPREHENSIVE RESULTS")
    print("="*100)
    print(f"{'Model Name':<50} {'RMSE':<10} {'R²':<10} {'Q²':<10} {'CV-RMSE':<10} {'Samples':<10}")
    print("-" * 100)
    
    for category in ['Low', 'Intermediate', 'High']:
        if category in results:
            for model_type, result in results[category].items():
                metrics = result['metrics']
                model_name = f"{category}_{model_type}"
                print(f"{model_name:<50} {metrics['rmse']:<10.4f} {metrics['r2']:<10.4f} {metrics['q2']:<10.4f} {result['cv_rmse']:<10.4f} {result['n_samples']:<10}")
    
    # Save comprehensive results
    results_summary_path = Path(save_dir) / "transfer_learning_summary.csv"
    summary_data = []
    
    for category in results:
        for model_type, result in results[category].items():
            summary_data.append({
                'Category': category,
                'Model_Type': model_type,
                'RMSE': result['metrics']['rmse'],
                'R2': result['metrics']['r2'],
                'Q2': result['metrics']['q2'],
                'CV_RMSE': result['cv_rmse'],
                'N_Samples': result['n_samples'],
                'Model_Path': result['final_model_path'],
                'Pretrained_From': result.get('pretrained_from', 'None'),
                'Freeze_Config': str(result['freeze_config'])
            })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(results_summary_path, index=False)
    print(f"\nSaved comprehensive summary to: {results_summary_path}")
    
    return results